# Simplified Object Detection

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/O-2wice/synthetic-object-detection/blob/main/notebooks/object-detection.ipynb)
![Runtime](https://img.shields.io/badge/runtime-CPU%20or%20GPU-blue)
![Framework](https://img.shields.io/badge/framework-PyTorch-orange)

Find one object in a cluttered scene: predict what it is and where it is.

The detector is built from scratch rather than assembled from a detection
library, so every part is visible: a shared convolutional backbone, one head for
the class and one for the box, and a composite objective that trains both at
once. A YOLOv8 baseline is trained on the same data at the end for scale.

The dataset is generated rather than collected. Each image places a single
object at a random position on procedurally drawn clutter, and the label records
its class and bounding box in normalized centre form, `cx cy w h`.

## Project Workflow

1. Generate a synthetic detection dataset from a seed.
2. Inspect the images against their labels.
3. Define a backbone with classification and box-regression heads.
4. Define a composite loss over class and box.
5. Train with validation monitoring, checkpointing and early stopping.
6. Plot the learning curves.
7. Evaluate with class accuracy, mean IoU and detection rate.
8. Look at predictions against ground truth.
9. Train a YOLOv8 baseline on the same data and compare.

## Why the Data Is Generated

The original coursework version scraped Google Images for backgrounds with
`icrawler` and pasted in character cut-outs downloaded from Drive. That cannot
be rerun: search results drift, the crawler breaks, and the assets are not in
the repository.

Generating instead makes the dataset a function of a seed. A split is
reproducible on any machine, splits provably cannot overlap, storage is zero,
and the object's exact position is known rather than estimated, so labels are
correct by construction.

To use real cut-outs instead, drop transparent PNGs named `0_*.png`, `1_*.png`
and `2_*.png` into `assets/objects/`. The placement, labelling and augmentation
logic is identical either way.

**Author:** Robert Ouko Oyombe

## Environment Setup

In [ ]:
import importlib.util
import json
import math
import random
import sys
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageDraw
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

# A fixed seed makes the dataset, the split boundaries and the weight
# initialisation repeatable, so two runs of this notebook are comparable.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    # find_spec imports the parent package, which does not exist off Colab.
    IN_COLAB = False

# Mount Drive when a browser is attached to the kernel. It gives the trained
# artifacts somewhere to land that is not wiped with the runtime. The mount
# fails without a browser, which is not fatal: paths fall back to the runtime.
DRIVE_DIR = None
if IN_COLAB:
    try:
        from google.colab import drive

        drive.mount("/content/drive")
        DRIVE_DIR = Path("/content/drive/MyDrive/synthetic-object-detection")
        DRIVE_DIR.mkdir(parents=True, exist_ok=True)
        print(f"Drive mounted: {DRIVE_DIR}")
    except Exception as exc:
        print(f"Drive not mounted ({type(exc).__name__}). Using runtime-local paths.")

OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = OUTPUT_DIR / "models"
METRIC_DIR = OUTPUT_DIR / "metrics"
ASSET_DIR = PROJECT_ROOT / "assets" / "objects"
for directory in (MODEL_DIR, METRIC_DIR):
    directory.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Runtime accelerator: {'GPU' if torch.cuda.is_available() else 'CPU'}")
print(f"Using device: {device}")
print(f"Project root: {PROJECT_ROOT.resolve()}")
print(f"Running in Colab: {IN_COLAB}")

## Generating the Dataset

Three object classes share a silhouette but differ in pattern and palette, so
the classification head has to attend to appearance rather than shape alone.
They are drawn on transparent canvases, rotated, then composited onto clutter.

The clutter matters. On a plain background the task collapses to finding the
only non-uniform region, and the detector would never have to learn anything
about the objects themselves.

In [ ]:
CLASS_NAMES = ["striped", "spotted", "chevron"]
NUM_CLASSES = len(CLASS_NAMES)

# Distinct palettes so the three classes are separable by colour as well as
# pattern. A model that only learned colour would still have to localise.
CLASS_PALETTES = [
    ((214, 40, 40), (255, 255, 255)),    # striped: red / white
    ((29, 53, 87), (241, 250, 238)),     # spotted: navy / off-white
    ((247, 181, 56), (61, 64, 91)),      # chevron: amber / slate
]

In [ ]:
@dataclass(frozen=True)
class Box:
    """Axis-aligned box in normalized centre form."""

    cx: float
    cy: float
    w: float
    h: float

    def to_corners(self) -> tuple[float, float, float, float]:
        return (self.cx - self.w / 2, self.cy - self.h / 2,
                self.cx + self.w / 2, self.cy + self.h / 2)

    def as_tuple(self) -> tuple[float, float, float, float]:
        return (self.cx, self.cy, self.w, self.h)

In [ ]:
def draw_object(rng: random.Random, class_id: int, size: int) -> Image.Image:
    """Draw one object on a transparent canvas.

    The three classes share a silhouette but differ in pattern and palette, so
    the classification head has to attend to appearance rather than shape alone.
    """
    primary, secondary = CLASS_PALETTES[class_id]
    canvas = Image.new("RGBA", (size, size), (0, 0, 0, 0))
    draw = ImageDraw.Draw(canvas)

    pad = size // 10
    body = (pad, pad, size - pad, size - pad)
    draw.ellipse(body, fill=primary + (255,))

    if class_id == 0:  # horizontal stripes
        step = max(3, size // 7)
        for offset in range(pad, size - pad, step * 2):
            draw.rectangle((pad, offset, size - pad, offset + step), fill=secondary + (255,))
    elif class_id == 1:  # scattered dots
        radius = max(2, size // 12)
        for _ in range(7):
            x = rng.randint(pad + radius, size - pad - radius)
            y = rng.randint(pad + radius, size - pad - radius)
            draw.ellipse((x - radius, y - radius, x + radius, y + radius),
                         fill=secondary + (255,))
    else:  # nested chevrons
        for k in range(3):
            inset = pad + k * max(3, size // 9)
            draw.line([(inset, size * 0.62), (size / 2, inset + size * 0.12),
                       (size - inset, size * 0.62)],
                      fill=secondary + (255,), width=max(2, size // 14))

    # Re-apply the ellipse as a mask so patterns cannot spill outside the body.
    mask = Image.new("L", (size, size), 0)
    ImageDraw.Draw(mask).ellipse(body, fill=255)
    canvas.putalpha(mask)
    return canvas

In [ ]:
def draw_background(rng: random.Random, width: int, height: int) -> Image.Image:
    """Draw cluttered doodle-style clutter for the object to hide in.

    Clutter matters: on a plain background the task collapses to finding the
    only non-uniform region, and the detector would not have to learn anything
    about the objects themselves.
    """
    background = Image.new("RGB", (width, height), (250, 249, 246))
    draw = ImageDraw.Draw(background)

    for _ in range(90):
        shape = rng.choice(("line", "circle", "rect", "arc"))
        colour = tuple(rng.randint(60, 235) for _ in range(3))
        x1, y1 = rng.randint(0, width), rng.randint(0, height)
        x2, y2 = x1 + rng.randint(-90, 90), y1 + rng.randint(-90, 90)
        box = (min(x1, x2), min(y1, y2), max(x1, x2), max(y1, y2))

        if shape == "line":
            draw.line((x1, y1, x2, y2), fill=colour, width=rng.randint(1, 4))
        elif shape == "circle":
            draw.ellipse(box, outline=colour, width=rng.randint(1, 4))
        elif shape == "rect":
            draw.rectangle(box, outline=colour, width=rng.randint(1, 4))
        else:
            start = rng.randint(0, 300)
            draw.arc(box, start=start, end=start + rng.randint(40, 300),
                     fill=colour, width=rng.randint(1, 4))

    return background

In [ ]:
def load_object_sprites(directory) -> list[Image.Image] | None:
    """Load user-supplied object cut-outs, if any are present.

    Drop transparent PNGs into `assets/objects/` named `0_*.png`, `1_*.png`,
    `2_*.png` (the prefix is the class id) and they replace the drawn shapes.
    This is the hook for running the pipeline on real cut-outs, such as the
    character sprites the original coursework used. When the directory is empty
    or missing, everything falls back to the procedural objects so the dataset
    still regenerates anywhere.
    """
    from pathlib import Path

    directory = Path(directory)
    if not directory.is_dir():
        return None

    sprites: list[Image.Image | None] = [None] * NUM_CLASSES
    for path in sorted(directory.glob("*.png")):
        try:
            class_id = int(path.name.split("_")[0])
        except ValueError:
            continue
        if 0 <= class_id < NUM_CLASSES and sprites[class_id] is None:
            sprites[class_id] = Image.open(path).convert("RGBA")

    return sprites if all(s is not None for s in sprites) else None

In [ ]:
def make_sample(seed: int, image_size: int = 224,
                min_scale: float = 0.16, max_scale: float = 0.34,
                sprites: list[Image.Image] | None = None
                ) -> tuple[Image.Image, int, Box]:
    """Build one (image, class_id, box) sample deterministically from `seed`.

    Pass `sprites` to composite real cut-outs instead of the drawn objects; the
    labelling, placement and augmentation logic is identical either way.
    """
    rng = random.Random(seed)

    background = draw_background(rng, image_size, image_size)
    class_id = rng.randrange(NUM_CLASSES)

    side = int(image_size * rng.uniform(min_scale, max_scale))
    if sprites is not None:
        sprite = sprites[class_id].copy()
        sprite.thumbnail((side, side), Image.LANCZOS)
    else:
        sprite = draw_object(rng, class_id, side)
    sprite = sprite.rotate(rng.uniform(-25, 25), resample=Image.BICUBIC, expand=True)

    # Crop back to the drawn pixels so the label matches the visible object
    # rather than the transparent canvas it was rotated inside.
    bbox = sprite.getbbox()
    sprite = sprite.crop(bbox)
    obj_w, obj_h = sprite.size

    x = rng.randint(0, max(0, image_size - obj_w))
    y = rng.randint(0, max(0, image_size - obj_h))
    background.paste(sprite, (x, y), sprite)

    box = Box(
        cx=(x + obj_w / 2) / image_size,
        cy=(y + obj_h / 2) / image_size,
        w=obj_w / image_size,
        h=obj_h / image_size,
    )
    return background, class_id, box

## Augmentation Has to Move the Label

This is the single most consequential correction to the original notebook.

The coursework version built its training transform like this:

```python
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(...),
])
```

Those transforms move the pixels. The bounding box in the label file does not
move with them, so on roughly half of every epoch the model was shown a mirrored
image and told the object was still on the original side. A flipped image scored
against its unflipped label overlaps it by about `0.12` IoU, so this was not
noise on the training signal; it was a wrong signal.

Any geometric augmentation therefore has to transform the label alongside the
image, which is why the flip lives in the dataset rather than in a `Compose`.

In [ ]:
def horizontal_flip(image: Image.Image, box: Box) -> tuple[Image.Image, Box]:
    """Mirror an image and its box together.

    The coursework applied RandomHorizontalFlip to the image alone and left the
    label untouched, so roughly half of every epoch taught the model a box on
    the wrong side. Any geometric augmentation has to move the label with the
    pixels, which is why it lives here rather than in a torchvision Compose.
    """
    return image.transpose(Image.FLIP_LEFT_RIGHT), Box(1.0 - box.cx, box.cy, box.w, box.h)

In [ ]:
def iou(a: Box, b: Box) -> float:
    """Intersection over union of two normalized centre-form boxes."""
    ax1, ay1, ax2, ay2 = a.to_corners()
    bx1, by1, bx2, by2 = b.to_corners()

    inter_w = max(0.0, min(ax2, bx2) - max(ax1, bx1))
    inter_h = max(0.0, min(ay2, by2) - max(ay1, by1))
    intersection = inter_w * inter_h

    union = a.w * a.h + b.w * b.h - intersection
    return intersection / union if union > 0 else 0.0

## Dataset and DataLoaders

Samples are generated on demand from a seed, so nothing is written to disk and
the three splits cannot overlap: each owns a disjoint seed range.

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

_to_tensor = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

class SyntheticDetectionDataset(Dataset):
    """Generate samples on the fly from a deterministic seed range.

    Nothing is written to disk. Each index maps to a fixed seed, so a split is
    reproducible, splits cannot overlap, and regenerating costs no storage.

    Augmentation lives here rather than in a torchvision `Compose` because a
    geometric transform has to move the label with the pixels. The coursework
    version put `RandomHorizontalFlip` in the Compose and left the box alone,
    which trained the model against mirrored targets half the time.
    """

    def __init__(self, seed_start: int, length: int, image_size: int = 224,
                 augment: bool = False, sprites=None):
        self.seed_start = seed_start
        self.length = length
        self.image_size = image_size
        self.augment = augment
        self.sprites = sprites

    def __len__(self) -> int:
        return self.length

    def __getitem__(self, idx: int):
        seed = self.seed_start + idx
        image, class_id, box = make_sample(
            seed, self.image_size, sprites=self.sprites
        )

        if self.augment:
            # Seeded off the sample index so an epoch is reproducible.
            if (seed * 2654435761) % 2 == 0:
                image, box = horizontal_flip(image, box)

        target_box = torch.tensor(box.as_tuple(), dtype=torch.float32)
        return _to_tensor(image), torch.tensor(class_id, dtype=torch.long), target_box

In [ ]:
IMAGE_SIZE = 224
BATCH_SIZE = 32

TRAIN_SIZE, VAL_SIZE, TEST_SIZE = 6000, 1000, 1000

# Disjoint seed ranges, spaced so the splits can grow without ever colliding.
SPLIT_SEEDS = {"train": 0, "val": 1_000_000, "test": 2_000_000}

# Real cut-outs are used when assets/objects/ holds them, otherwise the drawn
# objects are used, so the notebook runs with nothing downloaded.
SPRITES = load_object_sprites(ASSET_DIR)
print(f"Object source: {'assets/objects' if SPRITES else 'procedurally drawn'}")

train_dataset = SyntheticDetectionDataset(
    SPLIT_SEEDS["train"], TRAIN_SIZE, IMAGE_SIZE, augment=True, sprites=SPRITES
)
val_dataset = SyntheticDetectionDataset(
    SPLIT_SEEDS["val"], VAL_SIZE, IMAGE_SIZE, augment=False, sprites=SPRITES
)
test_dataset = SyntheticDetectionDataset(
    SPLIT_SEEDS["test"], TEST_SIZE, IMAGE_SIZE, augment=False, sprites=SPRITES
)

# Samples are drawn with PIL on the fly, which is CPU-bound and becomes the
# bottleneck on a GPU runtime. Colab is Linux and forks workers cheaply; Windows
# spawns them and cannot unpickle a Dataset defined inside a notebook.
NUM_WORKERS = 2 if IN_COLAB else 0
loader_options = {"num_workers": NUM_WORKERS, "pin_memory": device.type == "cuda"}
if NUM_WORKERS:
    loader_options["persistent_workers"] = True

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, **loader_options)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, **loader_options)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, **loader_options)

print(f"Train / val / test: {len(train_dataset):,} / {len(val_dataset):,} / {len(test_dataset):,}")
print(f"DataLoader workers: {NUM_WORKERS}")

## Inspecting the Data

Drawing the label on top of the image is the cheapest possible check that the
pipeline is correct. If boxes did not sit on objects here, nothing downstream
would be worth measuring.

In [ ]:
def show_samples(dataset, rows=2, cols=4):
    """Draw samples with their ground-truth boxes and class names."""
    fig, axes = plt.subplots(rows, cols, figsize=(3.1 * cols, 3.2 * rows))

    for ax, idx in zip(axes.flatten(), range(rows * cols)):
        image_tensor, class_id, box = dataset[idx]

        # Undo the ImageNet normalisation so the image displays correctly.
        image = image_tensor.permute(1, 2, 0).numpy()
        image = image * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
        ax.imshow(np.clip(image, 0, 1))

        cx, cy, w, h = box.tolist()
        side = image.shape[0]
        ax.add_patch(patches.Rectangle(
            ((cx - w / 2) * side, (cy - h / 2) * side), w * side, h * side,
            linewidth=2, edgecolor="lime", facecolor="none",
        ))
        ax.set_title(CLASS_NAMES[class_id], fontsize=10)
        ax.axis("off")

    plt.tight_layout()
    plt.show()


show_samples(train_dataset)

## The Detector

A ResNet-18 backbone feeds two heads: one predicts the class, the other
regresses the box. Both read the same features, so the backbone is trained by
both objectives at once.

Two departures from the coursework model, both about localization:

- **The neck keeps a coarse spatial grid.** The original pooled to `1x1`, which
  averages away *where* things are, and then asked a linear layer to regress
  `cx, cy` from that. Pooling to `3x3` preserves position at negligible cost.
- **The box head ends in a sigmoid.** Targets are normalized to `[0, 1]`. An
  unbounded head begins by predicting boxes that cannot exist and spends early
  training walking back into range.

In [ ]:
class SingleObjectDetector(nn.Module):
    """ResNet-18 backbone with a classification head and a box head.

    Two departures from the coursework model, both aimed at localization:

    - The neck pools to a 3x3 grid instead of 1x1. Collapsing the feature map
      to a single vector averages away *where* things are, which is precisely
      what the box head needs; keeping a coarse grid preserves it at negligible
      cost.
    - The box head ends in a sigmoid. Targets are normalized to [0, 1], so an
      unbounded head starts out predicting impossible boxes and spends early
      training walking back into range.
    """

    def __init__(self, num_classes: int = NUM_CLASSES, grid: int = 3,
                 pretrained: bool = True, freeze_backbone: bool = False):
        super().__init__()
        from torchvision import models

        weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = models.resnet18(weights=weights)
        self.backbone = nn.Sequential(*list(backbone.children())[:-2])

        if freeze_backbone:
            for parameter in self.backbone.parameters():
                parameter.requires_grad = False

        self.pool = nn.AdaptiveAvgPool2d((grid, grid))
        feature_dim = 512 * grid * grid

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(feature_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )
        self.box_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(feature_dim, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 4),
            nn.Sigmoid(),
        )

    def forward(self, x):
        features = self.pool(self.backbone(x))
        return self.classifier(features), self.box_head(features)

In [ ]:
model = SingleObjectDetector(pretrained=True).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(model.classifier)
print(f"Total parameters:     {total:,}")
print(f"Trainable parameters: {trainable:,}")

## Measuring Overlap

Intersection over union is the quantity everything downstream depends on, so it
is worth writing once, carefully, and reusing it for both the loss and the
metrics.

In [ ]:
def boxes_to_corners(boxes: torch.Tensor) -> torch.Tensor:
    """[N, 4] centre form -> [N, 4] corner form, batched."""
    cx, cy, w, h = boxes.unbind(-1)
    return torch.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], dim=-1)


def box_iou(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    """Elementwise IoU for two [N, 4] batches of centre-form boxes."""
    p = boxes_to_corners(pred)
    t = boxes_to_corners(target)

    inter_w = (torch.min(p[:, 2], t[:, 2]) - torch.max(p[:, 0], t[:, 0])).clamp(min=0)
    inter_h = (torch.min(p[:, 3], t[:, 3]) - torch.max(p[:, 1], t[:, 1])).clamp(min=0)
    intersection = inter_w * inter_h

    area_p = (pred[:, 2] * pred[:, 3]).clamp(min=0)
    area_t = (target[:, 2] * target[:, 3]).clamp(min=0)
    union = area_p + area_t - intersection
    return intersection / union.clamp(min=1e-9)

## The Loss

Detection trains two different kinds of prediction at once: a discrete class and
four continuous coordinates. The objective is therefore a weighted sum.

$$
\mathcal{L} = \lambda_{cls}\,\mathcal{L}_{CE}
             + \lambda_{L_1}\,\mathcal{L}_{smooth\,L_1}
             + \lambda_{IoU}\,(1 - \text{IoU})
$$

The coursework version used cross-entropy plus smooth L1 alone. Smooth L1 treats
each coordinate independently and is scale sensitive: an error of 0.05 barely
matters for a large box and ruins a small one. Adding `1 - IoU` optimises the
quantity actually reported, while keeping L1 preserves a useful gradient when
the boxes do not yet overlap and IoU is flat at zero.

In [ ]:
class DetectionLoss(nn.Module):
    """Cross-entropy for the class, plus a box term that mixes L1 and IoU.

    Smooth L1 alone treats every coordinate independently and is scale
    sensitive: the same absolute error matters far more for a small object than
    a large one. Adding (1 - IoU) optimises the quantity the metric actually
    reports, while the L1 term keeps gradients useful when boxes do not yet
    overlap and IoU is flat at zero.
    """

    def __init__(self, w_class: float = 1.0, w_l1: float = 5.0, w_iou: float = 2.0):
        super().__init__()
        self.w_class = w_class
        self.w_l1 = w_l1
        self.w_iou = w_iou

    def forward(self, class_logits, pred_boxes, true_classes, true_boxes):
        class_loss = F.cross_entropy(class_logits, true_classes)
        l1_loss = F.smooth_l1_loss(pred_boxes, true_boxes)
        iou_loss = (1.0 - box_iou(pred_boxes, true_boxes)).mean()

        total = self.w_class * class_loss + self.w_l1 * l1_loss + self.w_iou * iou_loss
        return total, {
            "class": class_loss.detach(),
            "l1": l1_loss.detach(),
            "iou": iou_loss.detach(),
        }

## Metrics

Every image holds exactly one object and the model emits exactly one prediction,
so the number of predictions equals the number of ground truths by construction.
Precision and recall are therefore the *same number*, and reporting them as
though they could differ hides errors rather than revealing them.

The original notebook reported precision `0.6450` alongside recall `1.0000`.
That gap was not a finding. Its filter for padding rows was
`true_classes[i] != 0`, and class 0 was a real class, so every image of that
class contributed a prediction but no ground truth. One third of the test set
was silently excluded from the denominator of recall but not of precision.

What is reported here instead:

- **class accuracy** — did it name the right object?
- **mean IoU** — how well does the predicted box overlap the true one?
- **detection rate** — the fraction where the class is right *and* IoU clears
  0.5, which is the single number that covers both heads.

In [ ]:
@torch.no_grad()
def evaluate(model, loader, device, iou_threshold: float = 0.5) -> dict:
    """Accuracy, mean IoU and detection rate over a loader.

    Every image holds exactly one object and the model emits exactly one
    prediction, so the number of predictions and the number of ground truths are
    equal by construction. Precision and recall are therefore the same number,
    and reporting them as if they could differ hides mistakes: the coursework
    version excluded class 0 from the ground-truth count but not from the
    prediction count, which is what produced its precision 0.645 / recall 1.000.

    A detection counts as correct only when the class is right *and* IoU clears
    the threshold, so the single figure covers both heads.
    """
    model.eval()
    total = 0
    class_correct = 0
    detected = 0
    iou_sum = 0.0
    per_class = {c: [0, 0] for c in range(NUM_CLASSES)}  # [correct, count]

    for images, classes, boxes in loader:
        images = images.to(device, non_blocking=True)
        classes = classes.to(device, non_blocking=True)
        boxes = boxes.to(device, non_blocking=True)

        logits, pred_boxes = model(images)
        predicted = logits.argmax(dim=1)
        ious = box_iou(pred_boxes, boxes)

        correct = (predicted == classes) & (ious >= iou_threshold)

        total += images.size(0)
        class_correct += (predicted == classes).sum().item()
        detected += correct.sum().item()
        iou_sum += ious.sum().item()

        for class_id in range(NUM_CLASSES):
            mask = classes == class_id
            per_class[class_id][0] += correct[mask].sum().item()
            per_class[class_id][1] += mask.sum().item()

    return {
        "samples": total,
        "class_accuracy": class_correct / max(total, 1),
        "mean_iou": iou_sum / max(total, 1),
        "detection_rate": detected / max(total, 1),
        "per_class_detection_rate": {
            c: (hits / count if count else float("nan"))
            for c, (hits, count) in per_class.items()
        },
    }

## Training

Each epoch writes two checkpoints: the best weights so far, and a resume file
carrying optimizer state and history. Colab runtimes disconnect, and the resume
file is what makes an interruption cost one epoch instead of the whole run.

In [ ]:
EPOCHS = 12
PATIENCE = 3
LEARNING_RATE = 1e-4
PROGRESS_EVERY = 50

criterion = DetectionLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

best_path = MODEL_DIR / "best_detector.pth"
last_path = MODEL_DIR / "last_detector.pth"

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    """One pass over `loader`. Trains when an optimizer is given, else evaluates.

    Losses are weighted by batch size so a short final batch cannot distort the
    epoch mean, which keeps the training and validation curves comparable.
    """
    training = optimizer is not None
    model.train(training)

    totals = {"total": 0.0, "class": 0.0, "l1": 0.0, "iou": 0.0}
    seen = 0

    with torch.set_grad_enabled(training):
        for batch_idx, (images, classes, boxes) in enumerate(loader, start=1):
            images = images.to(device, non_blocking=True)
            classes = classes.to(device, non_blocking=True)
            boxes = boxes.to(device, non_blocking=True)

            logits, pred_boxes = model(images)
            loss, parts = criterion(logits, pred_boxes, classes, boxes)

            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            batch = images.size(0)
            seen += batch
            totals["total"] += loss.item() * batch
            for key, value in parts.items():
                totals[key] += value.item() * batch

            if training and (batch_idx == 1 or batch_idx % PROGRESS_EVERY == 0
                             or batch_idx == len(loader)):
                print(f"  batch {batch_idx:04d}/{len(loader)} loss {loss.item():.4f}",
                      flush=True)

    return {key: value / seen for key, value in totals.items()}


def train(model, optimizer, criterion, train_loader, val_loader,
          epochs=EPOCHS, patience=PATIENCE, best_path=best_path, last_path=last_path):
    """Train the detector, resuming from `last_path` when it exists."""
    history = {"train": [], "val": []}
    start_epoch = 0
    best_val = float("inf")
    stale = 0

    if last_path.exists():
        checkpoint = torch.load(last_path, map_location=device)
        model.load_state_dict(checkpoint["model_state"])
        optimizer.load_state_dict(checkpoint["optimizer_state"])
        history = checkpoint["history"]
        start_epoch = checkpoint["epoch"]
        best_val = checkpoint["best_val"]
        stale = checkpoint["stale"]
        print(f"Resuming at epoch {start_epoch + 1}/{epochs}.", flush=True)

    if start_epoch >= epochs:
        print(f"Checkpoint already covers {epochs} epochs; nothing to do.")
        return history

    for epoch in range(start_epoch, epochs):
        started = time.time()
        print(f"Epoch {epoch + 1:02d}/{epochs}", flush=True)

        train_metrics = run_epoch(model, train_loader, criterion, optimizer)
        val_metrics = run_epoch(model, val_loader, criterion)

        history["train"].append(train_metrics)
        history["val"].append(val_metrics)

        if val_metrics["total"] < best_val:
            best_val = val_metrics["total"]
            stale = 0
            torch.save(model.state_dict(), best_path)
            status = f"validation improved, wrote {best_path.name}"
        else:
            stale += 1
            status = f"no improvement ({stale}/{patience})"

        torch.save({
            "epoch": epoch + 1,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "history": history,
            "best_val": best_val,
            "stale": stale,
        }, last_path)

        print(f"  train {train_metrics['total']:.4f} | val {val_metrics['total']:.4f} "
              f"| {(time.time() - started) / 60:.1f} min | {status}", flush=True)

        if stale >= patience:
            print(f"Early stopping after {patience} epochs without improvement.")
            break

    return history

In [ ]:
# Safe to re-run: resumes from the last completed epoch if it was interrupted.
history = train(model, optimizer, criterion, train_loader, val_loader)

In [ ]:
with (METRIC_DIR / "training_history.json").open("w", encoding="utf-8") as file:
    json.dump(history, file, indent=2)

# The complete history, echoed so it survives in the saved notebook even when a
# resumed run only logs the epochs it executed.
print("HISTORY_JSON_BEGIN")
print(json.dumps(history))
print("HISTORY_JSON_END")

model.load_state_dict(torch.load(best_path, map_location=device))
model.eval()
print(f"Loaded best checkpoint from {best_path.name}")

## Learning Curves

The composite loss is plotted alongside its parts, because a flat total can hide
one head still improving while the other has stopped.

In [ ]:
def plot_history(history):
    """Plot the total loss and each component for train and validation."""
    epochs = range(1, len(history["train"]) + 1)
    keys = ["total", "class", "l1", "iou"]

    fig, axes = plt.subplots(1, len(keys), figsize=(4.2 * len(keys), 3.6))
    for ax, key in zip(axes, keys):
        ax.plot(epochs, [m[key] for m in history["train"]], label="train")
        ax.plot(epochs, [m[key] for m in history["val"]], label="validation")
        ax.set_title(key)
        ax.set_xlabel("epoch")
        ax.grid(alpha=0.3)
    axes[0].set_ylabel("loss")
    axes[0].legend()

    plt.tight_layout()
    plt.show()


plot_history(history)

## Evaluation

In [ ]:
test_metrics = evaluate(model, test_loader, device)

with (METRIC_DIR / "test_results.json").open("w", encoding="utf-8") as file:
    json.dump(test_metrics, file, indent=2)

print(f"Test samples:    {test_metrics['samples']}")
print(f"Class accuracy:  {test_metrics['class_accuracy']:.4f}")
print(f"Mean IoU:        {test_metrics['mean_iou']:.4f}")
print(f"Detection rate:  {test_metrics['detection_rate']:.4f}  (class correct and IoU >= 0.5)")
print("Per class detection rate:")
for class_id, rate in test_metrics["per_class_detection_rate"].items():
    print(f"  {CLASS_NAMES[int(class_id)]:<10} {rate:.4f}")

## Predictions Against Ground Truth

Green is the label, red the prediction. The per-image IoU is printed above each
panel, which makes the failure modes legible: a box in the right place at the
wrong scale looks very different from one that found the wrong object entirely.

In [ ]:
@torch.no_grad()
def show_predictions(model, dataset, count=8):
    """Overlay predicted and ground-truth boxes on test images."""
    model.eval()
    cols = 4
    rows = math.ceil(count / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(3.3 * cols, 3.6 * rows))

    for ax, idx in zip(axes.flatten(), range(count)):
        image_tensor, class_id, true_box = dataset[idx]
        logits, pred_box = model(image_tensor.unsqueeze(0).to(device))

        predicted = int(logits.argmax(dim=1).item())
        pred_box = pred_box[0].cpu()
        overlap = box_iou(pred_box.unsqueeze(0), true_box.unsqueeze(0)).item()

        image = image_tensor.permute(1, 2, 0).numpy()
        image = np.clip(image * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN), 0, 1)
        ax.imshow(image)
        side = image.shape[0]

        for box, colour in ((true_box, "lime"), (pred_box, "red")):
            cx, cy, w, h = box.tolist()
            ax.add_patch(patches.Rectangle(
                ((cx - w / 2) * side, (cy - h / 2) * side), w * side, h * side,
                linewidth=2, edgecolor=colour, facecolor="none",
            ))

        mark = "correct" if predicted == class_id and overlap >= 0.5 else "missed"
        ax.set_title(f"{CLASS_NAMES[predicted]} / {CLASS_NAMES[class_id]}\n"
                     f"IoU {overlap:.2f} ({mark})", fontsize=9)
        ax.axis("off")

    plt.tight_layout()
    plt.show()


show_predictions(model, test_dataset)

## YOLOv8 Baseline

The custom detector is worth little without something to measure it against.
YOLOv8 is trained on the same objects and the same generator, so the comparison
isolates the modelling rather than the data.

The two are not equivalent by design. YOLO predicts a variable number of boxes
through anchors and non-maximum suppression, and arrives pretrained on COCO. The
model above predicts exactly one box because the task guarantees exactly one
object. The point of the comparison is scale: how much of the gap comes from
architecture, and how much from a task that was made easy.

Training YOLO needs the dataset on disk in its directory layout, so the
generator is written out here in YOLO format.

In [ ]:
%pip install -q ultralytics

In [ ]:
YOLO_ROOT = PROJECT_ROOT / "outputs" / "yolo_dataset"
YOLO_IMAGE_SIZE = 224
YOLO_SPLITS = {"train": (SPLIT_SEEDS["train"], 1500),
               "val": (SPLIT_SEEDS["val"], 300),
               "test": (SPLIT_SEEDS["test"], 300)}


def export_yolo_dataset(root=YOLO_ROOT, splits=YOLO_SPLITS, sprites=SPRITES):
    """Write the generated samples in the layout Ultralytics expects.

    Uses the same seeds as the tensors above, so YOLO trains on exactly the same
    images as the custom detector rather than a fresh draw.
    """
    for split, (seed_start, count) in splits.items():
        image_dir = root / "images" / split
        label_dir = root / "labels" / split
        image_dir.mkdir(parents=True, exist_ok=True)
        label_dir.mkdir(parents=True, exist_ok=True)

        for i in range(count):
            image, class_id, box = make_sample(
                seed_start + i, YOLO_IMAGE_SIZE, sprites=sprites
            )
            image.save(image_dir / f"{i:05d}.jpg", quality=92)
            (label_dir / f"{i:05d}.txt").write_text(
                f"{class_id} {box.cx:.6f} {box.cy:.6f} {box.w:.6f} {box.h:.6f}\n"
            )

    config = root / "data.yaml"
    config.write_text(
        f"path: {root.resolve()}\n"
        "train: images/train\n"
        "val: images/val\n"
        "test: images/test\n"
        f"nc: {NUM_CLASSES}\n"
        f"names: {CLASS_NAMES}\n"
    )
    print(f"Wrote YOLO dataset to {root}")
    return config


yolo_config = export_yolo_dataset()

In [ ]:
from ultralytics import YOLO

yolo = YOLO("yolov8n.pt")
yolo_results = yolo.train(
    data=str(yolo_config),
    epochs=25,
    imgsz=YOLO_IMAGE_SIZE,
    batch=32,
    project=str(PROJECT_ROOT / "outputs" / "yolo_runs"),
    name="detect",
    exist_ok=True,
    verbose=True,
)

In [ ]:
yolo_metrics = yolo.val(data=str(yolo_config), split="test")

yolo_summary = {
    "map50": float(yolo_metrics.box.map50),
    "map50_95": float(yolo_metrics.box.map),
    "precision": float(yolo_metrics.box.mp),
    "recall": float(yolo_metrics.box.mr),
}

with (METRIC_DIR / "yolo_results.json").open("w", encoding="utf-8") as file:
    json.dump(yolo_summary, file, indent=2)

print("YOLOv8n on the held-out split:")
for key, value in yolo_summary.items():
    print(f"  {key:<10} {value:.4f}")

## Comparison

The two models report different native metrics, so they are compared on the one
they share: the fraction of test images where the object is found with the right
class at IoU >= 0.5.

In [ ]:
comparison = {
    "custom_detector": {
        "detection_rate_iou50": test_metrics["detection_rate"],
        "class_accuracy": test_metrics["class_accuracy"],
        "mean_iou": test_metrics["mean_iou"],
    },
    "yolov8n": yolo_summary,
}

with (METRIC_DIR / "comparison.json").open("w", encoding="utf-8") as file:
    json.dump(comparison, file, indent=2)

print("COMPARISON_JSON_BEGIN")
print(json.dumps(comparison))
print("COMPARISON_JSON_END")

## Saving Results

Everything under `/content` is discarded when the runtime is recycled, so the
metrics and checkpoints have to leave before the session ends. With Drive
mounted the archive is copied there; otherwise the path on the runtime is
reported so it can be collected manually.

In [ ]:
import shutil


def save_outputs(output_dir=OUTPUT_DIR, drive_dir=DRIVE_DIR):
    """Archive the results and report, honestly, where they ended up."""
    if not any(output_dir.rglob("*")):
        print(f"Nothing to archive: {output_dir} is empty.")
        return None

    staging = Path("/tmp/detection-artifacts") if IN_COLAB else output_dir.parent / "_staging"
    if staging.exists():
        shutil.rmtree(staging)
    # The resume checkpoint also carries optimizer state, which triples its size
    # and is of no use once training has finished.
    shutil.copytree(output_dir, staging,
                    ignore=shutil.ignore_patterns("last_*", "yolo_dataset", "yolo_runs"))

    archive = Path(shutil.make_archive(
        str(Path.cwd() / "object-detection-outputs"), "zip", staging))
    shutil.rmtree(staging, ignore_errors=True)
    print(f"Archived {archive.name} ({archive.stat().st_size / 1024 ** 2:.1f} MB)")

    if drive_dir is not None:
        destination = Path(drive_dir) / archive.name
        shutil.copy2(archive, destination)
        print(f"Copied to Drive: {destination}")
        return destination

    if not IN_COLAB:
        print(f"Archive written to: {archive}")
        return archive

    # files.download() only works with a Colab browser tab attached. From the
    # VS Code extension it queues JavaScript that never runs, so it cannot be
    # treated as success.
    try:
        from google.colab import files

        files.download(str(archive))
        print("Requested a browser download.")
        print("If no download appeared, this kernel has no browser attached.")
    except Exception as exc:
        print(f"Browser download unavailable ({type(exc).__name__}).")

    print(f"\nThe archive is on the runtime at: {archive}")
    return archive


save_outputs()